# 09 — Model Training

Trains baseline regression models and saves the fitted models and performance
table.

In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "configs").exists():
            return candidate
    raise FileNotFoundError(
        "Project root was not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
CONFIG_DIR = PROJECT_ROOT / "configs"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

for folder in [INTERIM_DIR, PROCESSED_DIR, OUTPUT_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression

data = pd.read_csv(PROCESSED_DIR / "station_samples" / "station_raster_samples.csv")

target_candidates = ["rainfall_mm", "rainfall", "precipitation"]
target = next((c for c in target_candidates if c in data.columns), None)
if target is None:
    raise ValueError("Rainfall target column was not found.")

exclude = {
    target, "station", "date", "year", "month",
    "latitude", "longitude"
}
features = [
    c for c in data.select_dtypes(include="number").columns
    if c not in exclude
]

X = data[features].replace([np.inf, -np.inf], np.nan)
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

models = {
    "linear_regression": LinearRegression(),
    "random_forest": RandomForestRegressor(
        n_estimators=300, random_state=42, n_jobs=-1
    ),
    "gradient_boosting": GradientBoostingRegressor(random_state=42),
}

In [ ]:
results = []
MODEL_DIR.mkdir(parents=True, exist_ok=True)

for name, estimator in models.items():
    pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", estimator),
    ])
    pipeline.fit(X_train, y_train)
    prediction = pipeline.predict(X_test)

    results.append({
        "model": name,
        "rmse": mean_squared_error(y_test, prediction) ** 0.5,
        "mae": mean_absolute_error(y_test, prediction),
        "r2": r2_score(y_test, prediction),
    })

    joblib.dump(
        {"pipeline": pipeline, "features": features, "target": target},
        MODEL_DIR / f"{name}.joblib",
    )

results = pd.DataFrame(results).sort_values("rmse")
results.to_csv(PROCESSED_DIR / "model_tables" / "training_results.csv", index=False)
display(results)